# Transformer Foundations, Part 4A: Why a Decoder-Only Model Works

## Guiding question

How can one stack learn every next-token lesson in parallel without peeking at the answers, then generate one token at a time without changing the objective?

Parts 1-3 gave Riverside's tokens identity, order, contextual routing, and a complete shape-preserving block. The unresolved complaint was architectural: a normal self-attention table can see both directions. A language model graded after `aria heard` must not inspect `the signal` first.

| Step | Visible failure | Smallest repair | Evidence |
|---|---|---|---|
| 1 | Full attention leaks future answers | Causal score mask | The forbidden triangle receives zero probability |
| 2 | One sentence looks like one lesson | Shift inputs and targets | Every valid position becomes one classification row |
| 3 | Batches need padding | Separate three mask jobs | Attention, padding, and loss exclusion remain distinct |
| 4 | Training is parallel; generation is serial | Feed choices back one at a time | Prefix work grows unless it is cached |
| 5 | Greedy output is brittle | Temperature and top-k | One logit vector yields controlled alternatives |

Keep [the handwritten theory companion](04-decoder-only-language-model-theory.md) beside this notebook. Part 4B turns these contracts into a readable PyTorch implementation.

**Scope:** this notebook builds and measures the causal objective. It does not repeat the complete block from Part 3 or claim that a tiny corpus produces a useful assistant.


In [ ]:
# Imports and deterministic teaching state
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.random.seed(404)
tokens = ["<bos>", "aria", "heard", "the", "signal", "<eos>"]
print("Running example:", " ".join(tokens))


## 1. The dishonest shortcut

**Predict:** If the state after `heard` can read `signal`, will low next-token loss prove that the model learned to predict?

No. It may only prove that the answer was visible. The smallest repair is a lower-triangular visibility rule: row `i` may read columns `0..i`, never columns after `i`.


In [ ]:
# Static causal evidence: allowed positions survive, future positions disappear
length = len(tokens)
allowed = np.tril(np.ones((length, length), dtype=bool))
raw_scores = np.linspace(-0.8, 1.0, length * length).reshape(length, length)
masked_scores = np.where(allowed, raw_scores, -np.inf)
shifted = masked_scores - np.max(masked_scores, axis=-1, keepdims=True)
attention = np.exp(shifted)
attention = attention / attention.sum(axis=-1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, matrix, title in zip(
    axes,
    [allowed.astype(float), attention],
    ["Legal reads (1 = visible)", "Probability after causal masking"],
):
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(length), tokens, rotation=45, ha="right")
    ax.set_yticks(range(length), tokens)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    ax.set_title(title)
fig.colorbar(image, ax=axes, shrink=0.75)
plt.tight_layout()
plt.show()

future_mass = attention[np.triu_indices(length, k=1)].sum()
print(f"Total probability assigned to future positions: {future_mass:.1f}")
assert future_mass == 0.0


#### That fixed leakage, but three different things are called a mask

| Contract | Where it acts | What it blocks | Typical shape |
|---|---|---|---|
| Causal attention mask | attention score table | future key positions | `(time, time)` |
| Padding key mask | attention score table | placeholder keys in this batch | `(batch, time)` |
| Ignored loss target | cross-entropy rows | targets that are not real lessons | `(batch, time)` |

A causal mask cannot remove padded targets from the loss. `ignore_index` cannot stop attention from reading a padded key. Similar names do not make them interchangeable.


In [ ]:
# Shift once, then expose the three contracts on a padded two-example batch
examples = [
    ["<bos>", "aria", "heard", "the", "signal", "<eos>"],
    ["<bos>", "aria", "returned", "<eos>"],
]
width = max(len(row) - 1 for row in examples)
input_rows, target_rows, padding_rows = [], [], []
for row in examples:
    inputs, targets = row[:-1], row[1:]
    pad_count = width - len(inputs)
    input_rows.append(inputs + ["<pad>"] * pad_count)
    target_rows.append(targets + ["IGNORE"] * pad_count)
    padding_rows.append([False] * len(inputs) + [True] * pad_count)

print("inputs:")
for row in input_rows:
    print(" ", row)
print("targets:")
for row in target_rows:
    print(" ", row)
print("padding-key mask (True means block):", padding_rows[1])
print("causal mask is shared across the batch:", (width, width))


## 2. One known sequence becomes many lessons

```text
visible input                    expected next token
<bos>                            aria
<bos> aria                       heard
<bos> aria heard                 the
<bos> aria heard the             signal
<bos> aria heard the signal      <eos>
```

Training can score those rows in one forward pass because the causal mask preserves the same information boundary each serial prediction would have had.

Generation is different. The next input depends on the model's last choice, so the loop must append one selected token before asking again.


In [ ]:
# Measured recomputation complaint and a static storyboard for the cache
max_new_tokens = 6
prompt_length = 4
recomputed_positions = [
    prompt_length + step for step in range(max_new_tokens)
]
cached_positions = [prompt_length] + [1] * (max_new_tokens - 1)

print("positions processed without cache:", recomputed_positions)
print("positions processed with cache:   ", cached_positions)
print("total without cache:", sum(recomputed_positions))
print("total with cache:   ", sum(cached_positions))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].bar(range(max_new_tokens), recomputed_positions, color="#c44e52")
axes[1].bar(range(max_new_tokens), cached_positions, color="#4c72b0")
axes[0].set_title("Recompute the whole prefix")
axes[1].set_title("Prime once, then append one state")
for ax in axes:
    ax.set_xlabel("generation step")
    ax.set_ylabel("positions processed")
plt.tight_layout()
plt.show()


In [ ]:
# Animation: the same measured work counts accumulate over generation
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.set_xlim(-0.5, max_new_tokens - 0.5)
ax.set_ylim(0, max(recomputed_positions) + 1)
ax.set_xlabel("generation step")
ax.set_ylabel("positions processed now")
bars_a = ax.bar(
    np.arange(max_new_tokens) - 0.18,
    np.zeros(max_new_tokens),
    width=0.36,
    label="recompute",
    color="#c44e52",
)
bars_b = ax.bar(
    np.arange(max_new_tokens) + 0.18,
    np.zeros(max_new_tokens),
    width=0.36,
    label="cache",
    color="#4c72b0",
)
ax.legend()

def update(frame):
    for index, bar in enumerate(bars_a):
        bar.set_height(recomputed_positions[index] if index <= frame else 0)
    for index, bar in enumerate(bars_b):
        bar.set_height(cached_positions[index] if index <= frame else 0)
    ax.set_title(f"Prefix grows through generation step {frame + 1}")
    return [*bars_a, *bars_b]

animation = FuncAnimation(fig, update, frames=max_new_tokens, interval=550, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))


#### The cache saves work; it does not change the distribution

Each layer stores earlier keys and values. The new token contributes one new query, key, and value. With the same weights, arithmetic, prefix, and selection rule, cached and full-prefix logits should agree within numerical tolerance. Part 4B tests that parity directly.

## 3. One score vector, several decoding policies

Greedy decoding always selects the largest logit. Sampling first reshapes the same scores:

- temperature below `1` sharpens preferences;
- temperature above `1` flattens them;
- top-k sets every candidate outside the best `k` to zero probability.

These controls change selection, not what the model learned.


In [ ]:
# One changed variable: temperature on the same four next-token logits
labels = ["signal", "map", "harbor", "lantern"]
logits = np.array([2.8, 1.6, 0.8, -0.2])

def probabilities(temperature):
    scaled = logits / temperature
    values = np.exp(scaled - scaled.max())
    return values / values.sum()

table = np.stack([probabilities(0.5), probabilities(1.0), probabilities(1.8)])
for temperature, row in zip([0.5, 1.0, 1.8], table):
    pairs = ", ".join(f"{label}={value:.3f}" for label, value in zip(labels, row))
    print(f"T={temperature:.1f}: {pairs}")

top_two = np.argsort(logits)[-2:]
print("top-k=2 keeps:", [labels[index] for index in top_two])


## Part 4A checkpoint

| Question | Evidence |
|---|---|
| Why causal masking? | Future-position probability measured exactly `0` |
| Why shift targets? | One sentence exposed five aligned next-token lessons |
| Why three mask contracts? | Score visibility, padded keys, and invalid targets act at different boundaries |
| Why serial generation? | Every new input depends on the prior selected token |
| Why cache? | The measured storyboard replaced repeated prefix work with one appended state |

**Built and measured:** causal visibility, aligned targets, mask distinctions, recomputation cost, temperature.

**Explained or illustrated:** top-k, stop tokens, per-layer KV cache.

**Named and deferred:** beam search and production cache layouts; they change search or systems engineering, not the causal objective.

**Unresolved question:** can a compact PyTorch model preserve every one of these contracts at once? Continue to [Part 4B: Decoder-Only PyTorch Lab](04b-decoder-only-language-model-lab.ipynb).
